# A notebook for fine-tuning fingerprint model generated after transfer learning of mobilenet pre-trained model 

## Project Pre-requisites

In [1]:
# Import libraries
import os
from pathlib import Path

import numpy as np
import pandas as pd

from PIL import Image
import matplotlib.pyplot as plt

# TensorFlow and Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks, saving

from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.utils import image_dataset_from_directory

# pre-trained model selection
MobileNet_version = "MobileNetV2"
fingerprint_model_name = MobileNet_version + "_FingerPrint_Model"

if MobileNet_version == "MobileNet":
    from tensorflow.keras.applications import MobileNet
    from tensorflow.keras.applications.mobilenet import preprocess_input
elif MobileNet_version == "MobileNetV2":
    from tensorflow.keras.applications import MobileNetV2
    from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
elif MobileNet_version == "MobileNetV3Small":
    from tensorflow.keras.applications import MobileNetV3Small
    from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
elif MobileNet_version == "MobileNetV3Large":
    from tensorflow.keras.applications import MobileNetV3Large
    from tensorflow.keras.applications.mobilenet_v3 import preprocess_input
else:
    print("Unknown MobileNet version name")
    exit(0)

In [2]:
# Check selected pre-trained model
print(f"Selected pre-trained model version : {MobileNet_version}")
print(f"Fingerprint model name : {fingerprint_model_name}")

Selected pre-trained model version : MobileNetV2
Fingerprint model name : MobileNetV2_FingerPrint_Model


In [3]:
# Configuration GPU
try:
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ GPU(s) détecté(s): {len(gpus)} - Croissance mémoire activée")
    else:
        print("⚠️  Aucun GPU détecté - Utilisation du CPU (entraînement sera lent)")
except Exception as e:
    print(f"Configuration GPU: {e}")

print(f"\n📦 Versions des bibliothèques :")
print(f"  - TensorFlow : {tf.__version__}")
print(f"  - Keras      : {keras.__version__}")
print(f"  - NumPy      : {np.__version__}")

print(f"\n🚀 Prêt pour le transfer learning en computer vision !")

✅ GPU(s) détecté(s): 1 - Croissance mémoire activée

📦 Versions des bibliothèques :
  - TensorFlow : 2.16.2
  - Keras      : 3.13.2
  - NumPy      : 1.26.4

🚀 Prêt pour le transfer learning en computer vision !


## Project Constants

In [4]:
# Random seeding 
TENSORFLOW_SEED = 42
tf.random.set_seed(TENSORFLOW_SEED)

NUMPY_SEED = 42
np.random.seed(NUMPY_SEED)

# Dataset configuration

DATASET_ROOTDIR_PATH = "/Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Training" 
print(f"📁 Fingerprints dataset images root directory path : {DATASET_ROOTDIR_PATH}")

# labels
# class_names = ['thumb', 'index', 'middle', 'ring', 'little']
class_names = ['thumb', 'index', 'middle', 'ring', 'little']

num_classes = len(class_names)
print(f"\n👁️ fingerprints classes : {class_names}")

# model name
fingerprint_model_basename = fingerprint_model_name + "_" + str(num_classes) + "_classes"
print(f"Adjusted fingerprint model name : {fingerprint_model_basename}")

📁 Fingerprints dataset images root directory path : /Users/laurent/Projects/projet-fingerprint-validator/dataset/SOCOFing_Training

👁️ fingerprints classes : ['thumb', 'index', 'middle', 'ring', 'little']
Adjusted fingerprint model name : MobileNetV2_FingerPrint_Model_5_classes


In [5]:
print("🔧 Datasets creation for training and validation ...\n")

# Dataset creation parameters
BATCH_SIZE = 64
IMAGE_SIZE = (96, 96)

VALIDATION_SPLIT = 0.2
DATASET_SEED = 42

# Create traning dataset with class filtering
train_ds = image_dataset_from_directory(
    DATASET_ROOTDIR_PATH,
    validation_split=VALIDATION_SPLIT,
    subset="training",
    seed=DATASET_SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',  # One-hot encoding for multi-class
    class_names=class_names  # Explicitly specify the class names to ensure correct mapping
)

# Create validation dataset with the same class names to ensure correct mapping
val_ds = image_dataset_from_directory(
    DATASET_ROOTDIR_PATH,
    validation_split=VALIDATION_SPLIT,
    subset="validation",
    seed=DATASET_SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    class_names=class_names  # Same order of classes
)

print(f"✅ Datasets created !\n")

print(f"📊 Training and validation datasets information:")
print(f"  - Training Dataset Classes             : {train_ds.class_names}")
print(f"  - Validation Dataset Classes           : {val_ds.class_names}")

# Checking number of classes in both datasets
if len(train_ds.class_names) != num_classes:
    print(f"\n⚠️  Error : {len(train_ds.class_names)} classes found instead of {num_classes} !")
    print(f"   Classes found: {train_ds.class_names}\n")
elif len(val_ds.class_names) != num_classes:
    print(f"\n⚠️  Error : {len(val_ds.class_names)} classes found au lieu de {num_classes} !")
    print(f"   Classes found : {val_ds.class_names}")
else:
    print(f"  - ✅ Number of correct classes : {num_classes} \n")

print(f"  - Training batches    : {len(train_ds)}")
print(f"  - Validation batches    : {len(val_ds)}")
print(f"  - Batch size            : {BATCH_SIZE}")
print(f"  - Image size            : {IMAGE_SIZE}\n")
print(f"  - Training samples      : ~{len(train_ds) * BATCH_SIZE}")
print(f"  - Validation samples    : ~{len(val_ds) * BATCH_SIZE}")

🔧 Datasets creation for training and validation ...

Found 23931 files belonging to 5 classes.
Using 19145 files for training.


2026-03-21 22:27:21.783720: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-03-21 22:27:21.783766: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-03-21 22:27:21.783778: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-03-21 22:27:21.783805: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-21 22:27:21.783839: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 23931 files belonging to 5 classes.
Using 4786 files for validation.
✅ Datasets created !

📊 Training and validation datasets information:
  - Training Dataset Classes             : ['thumb', 'index', 'middle', 'ring', 'little']
  - Validation Dataset Classes           : ['thumb', 'index', 'middle', 'ring', 'little']
  - ✅ Number of correct classes : 5 

  - Training batches    : 300
  - Validation batches    : 75
  - Batch size            : 64
  - Image size            : (96, 96)

  - Training samples      : ~19200
  - Validation samples    : ~4800


## Dataset Preparation : normalization and shuffling (training only), no augmentation

In [6]:
print("🎨 Fingerprint dataset preparation : normalisation and shuffling (training only) ...\n")

# Dataset preparation function
def prepare_dataset(ds, shuffle=True):

    # normalisation required for MobileNet
    normalization = lambda x, y: (preprocess_input(x), y)
    ds = ds.map(normalization, num_parallel_calls=tf.data.AUTOTUNE)
    
    # Shuffle (training only)
    if shuffle:
        ds = ds.shuffle(buffer_size=1000)
    
    # Performances optimisation
    ds = ds.cache()  # Memory caching
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)  # Prefetch for acceleration
    
    return ds

# Apply transformations
train_ds_prepared = prepare_dataset(train_ds, shuffle=True)
val_ds_prepared = prepare_dataset(val_ds, shuffle=False)

print("✅ Dataset preparation completed !\n")

print("📝 Applied transformations :")
print("  - Training dataset   : normalisation + shuffle")
print("  - Validation dataset : Normalisation")
print("\n  - Caching et prefetch activated for performance")

🎨 Fingerprint dataset preparation : normalisation and shuffling (training only) ...

✅ Dataset preparation completed !

📝 Applied transformations :
  - Training dataset   : normalisation + shuffle
  - Validation dataset : Normalisation

  - Caching et prefetch activated for performance


## Loading Frozen Previously Trained model with Pre-Trained MobileNet base model 

In [ ]:
fingerprint_model_dir = "/Users/laurent/Projects/projet-fingerprint-validator/models"
fingerprint_model_name = fingerprint_model_basename + "_Transfer_Learning.keras"
fingerprint_model_path = fingerprint_model_dir + "/" + fingerprint_model_name

print(f"📥 Loading Frozen Previously Trained {fingerprint_model_name} model ...")

# Loading frozen fingerprint model
try:
    frozen_fingerprint_model = tf.keras.models.load_model(fingerprint_model_path, compile=False)
except Exception as e:
    print(f"Unable to load pre-trained model {fingerprint_model_path} : {e}")
    exit(0)

print(f"✅ {fingerprint_model_name} model loaded !\n")

# Display model structure
print(f"📋 Model Structure - {fingerprint_model_name} :")
print(f"  - Input shape    : {frozen_fingerprint_model.input_shape}")
print(f"  - Output shape   : {frozen_fingerprint_model.output_shape}")
print(f"  - Number of layers : {len(frozen_fingerprint_model.layers)}")

# Count parameters
total_params = frozen_fingerprint_model.count_params()
print(f"\n  - Total number of parameters : {total_params:,}")
print(f"  - Memory size (approx.) : {total_params * 4 / (1024**2):.1f} MB")

📥 Loading Frozen Previously Trained MobileNetV2_FingerPrint_Model_5_classes.keras model ...
✅ MobileNetV2_FingerPrint_Model_5_classes.keras model loaded !

📋 Model Structure - MobileNetV2_FingerPrint_Model_5_classes.keras :
  - Input shape    : (None, 96, 96, 3)
  - Output shape   : (None, 5)
  - Number of layers : 6

  - Total number of parameters : 2,587,205
  - Memory size (approx.) : 9.9 MB


In [12]:
print(f"🔍 {fingerprint_model_name} pre-trained model architecture\n")
print("="*60)

def get_output_shape(layer):
    try:
        if hasattr(layer, 'output_shape'):
            return str(layer.output_shape)
        elif hasattr(layer, 'output'):
            return str(layer.output.shape)
        else:
            return "N/A"
    except:
        return "N/A"

# Display layers
print("\n📌 First layers (low-level feature extraction) :")
for i, layer in enumerate(frozen_fingerprint_model.layers):
    output_shape = get_output_shape(layer)
    print(f"  {i+1:3d}. {layer.name:40s} | Output: {output_shape:30s} | Trainable: {layer.trainable}")

print("\n      ...")

print("\n" + "="*60)

🔍 MobileNetV2_FingerPrint_Model_5_classes.keras pre-trained model architecture


📌 First layers (low-level feature extraction) :
    1. input_layer_1                            | Output: (None, 96, 96, 3)              | Trainable: True
    2. mobilenetv2_1.00_96                      | Output: (None, 1280)                   | Trainable: False
    3. dropout                                  | Output: (None, 1280)                   | Trainable: True
    4. dense                                    | Output: (None, 256)                    | Trainable: True
    5. dropout_1                                | Output: (None, 256)                    | Trainable: True
    6. dense_1                                  | Output: (None, 5)                      | Trainable: True

      ...



In [14]:
frozen_fingerprint_model.summary()

Model: "MobileNetV2_FingerPrint_Model_5_classes"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_96             │ (None, 1280)           │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,587,205 (9.87 MB)

 Trainable params: 329,221 (1.26 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Fine-tuning model training 

In [ ]:
# Constants for training callbacks
EARLYSTOPPING_PATIENCE = 30
EARLYSTOPPING_VERBOSE = 1

REDUCELRONPLATEAU_FACTOR = 0.5
REDUCELRONPLATEAU_PATIENCE = 4
REDUCELRONPLATEAU_MIN_LR = 1e-7
REDUCELRONPLATEAU_VERBOSE = 1

fingerprint_model_name = MobileNet_version + "_FingerPrint_Model_" + str(num_classes) + "_classes_Fine_Tuning.keras"
MODELCHECKPOINT_PATH = fingerprint_model_dir + "/" + fingerprint_model_name
MODELCHECKPOINT_VERBOSE = 1

FIT_NUM_EPOCHS = 120
FIT_VERBOSE = 1